In [ ]:
from src.graph import construct_amwn
from src.sym.dsl import Variable, CounterfactualTerm
import networkx as nx

sfm_markovian = build_sfm(
    sensitive_attr="S2_gender",
    outcome_attr="T_income",
    confounder_attrs=["relationship"],
    mediator_attrs=["hours-per-week"],
    latents=[
        ("U_X_W", ["S2_gender", "relationship"]),
    ],
)

# 2. Define Variables
X = Variable("S2_gender")
Y = Variable("T_income")
Z = Variable("relationship")
W = Variable("hours-per-week")
U_X_W = Variable("U_X_W")

term_Y_x = Y @ {X: 0}
term_Xp = X @ {}
term_Z = Z @ {}

Y_star = [term_Y_x, term_Xp]

# 4. Construct the AMWN
G_A = construct_amwn(sfm_markovian, Y_star)

print(nx.is_d_separator(G_A, term_Y_x, term_Xp, term_Z))

In [ ]:
# 1. Define the original causal diagram G
G = nx.DiGraph()
for var in ["W", "X", "Y", "Z"]:
    G.add_node(var, category="endogenous")

G.add_edges_from([("W", "Z"), ("Z", "X"), ("X", "Y"), ("Z", "Y")])

G.add_node("U_ZX", category="latent")
G.add_edges_from([("U_ZX", "Z"), ("U_ZX", "X")])

# 2. Define Variables
X = Variable("X")
Y = Variable("Y")
Z = Variable("Z")
W = Variable("W")
U = Variable("U_ZX")

# 3. Define the query counterfactual variables: Y_{x,w}, W_{x'}, X, Z_{x'}
term_Y_xw = Y @ {X: 0, W: 0}
term_X = X @ {}  # X factual
term_Z = Z @ {}
term_W = W @ {}

Y_star: list[Unknown | CounterfactualTerm] = [
    term_Y_xw,
    term_X,
    term_Z,
    term_W,
]

# 4. Construct the AMWN
G_A = construct_amwn(G, Y_star)

# Print Nodes in the generated AMWN
for node in G_A.nodes:
    print(node)